# VulnAuditAgent - 智能代码漏洞审计助手

## 项目简介
基于 HelloAgents 框架的三通道协同代码安全审计智能体：规则引擎（AST）+ 污点追踪（source→sink）+ LLM 研判（误报过滤）。

## 作者信息
- GitHub: @smdcdz
- 日期: 2026-09


## 1. 环境配置

In [ ]:
# 安装依赖
# !pip install -q hello-agents python-dotenv

import os
import sys
from dotenv import load_dotenv

# 加载 .env 中的 LLM 配置（参考 .env.example）
load_dotenv()
sys.path.insert(0, 'src')

## 2. 工具定义
三个自定义工具在 `src/audit_tools.py` 中实现：
- `RuleEngineTool`：规则引擎（AST 危险函数 + 正则硬编码密钥）
- `TaintTrackTool`：污点追踪（外部输入 → 危险函数链路）
- `LLMVerifyTool`：LLM 研判（真阳性/误报过滤 + 修复建议）

In [ ]:
from audit_tools import RuleEngineTool, TaintTrackTool, LLMVerifyTool
from hello_agents import SimpleAgent, HelloAgentsLLM, ToolRegistry

## 3. 智能体构建

In [ ]:
SYSTEM_PROMPT = '你是一名资深安全代码审计员，工作流为三通道协同：\n\n【第一步】调用 rule_scan 对源码做规则引擎扫描（危险函数/硬编码密钥/弱加密）\n【第二步】调用 taint_track 做污点追踪（外部输入->危险函数的数据流链路）\n【第三步】对规则扫描中 严重/高危 的每一条，调用 llm_verify 研判真伪\n【第四步】汇总输出 Markdown 格式的审计报告\n\n报告格式：\n# 代码审计报告\n## 概览\n| 项目 | 值 |\n表格：文件、扫描规则命中数、污点链路数、研判后确认漏洞数\n## 漏洞详情\n每条漏洞：\n### [序号] 漏洞类型（文件名:行号）\n- 风险等级:\n- 研判结论: 真阳性/误报/存疑\n- 代码片段:\n- 风险说明:\n- 修复建议:\n## 总结\n一段话总结整体安全状况和最优先修复项。\n\n注意：文件名只写相对文件名（如 vulnerable_sample.py），不要写完整路径。'


## 4. 功能演示
对示例漏洞文件 `data/vulnerable_sample.py` 执行完整审计：

In [ ]:
prompt = (
    '请审计项目中的 data/vulnerable_sample.py 文件。\n'
    '严格按照工作流执行：\n'
    '1. 调用工具 rule_scan，参数 file_path=data/vulnerable_sample.py\n'
    '2. 调用工具 taint_track，参数 file_path=data/vulnerable_sample.py\n'
    '3. 对规则扫描中 严重/高危 的每一条，调用 llm_verify 研判\n'
    '4. 汇总输出完整 Markdown 审计报告\n'
)

report = agent.run(prompt, max_tool_iterations=15)
print(report)

## 5. 保存报告

In [ ]:
os.makedirs('outputs', exist_ok=True)
with open('outputs/audit_report.md', 'w', encoding='utf-8') as f:
    f.write(report)
print('报告已保存: outputs/audit_report.md')

## 6. 总结
- 规则引擎提供确定性覆盖，污点追踪补盲区（如 SQL 注入拼接），LLM 研判过滤误报
- 三通道交叉验证，比单一手段更接近真实审计工作流
- 可扩展：新增规则只需在 SINK_RULES 加一行；sink/source 集合同理